In [2]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

import statsmodels as sms

In [3]:
data = pd.read_csv('../data/processed/cleaned_healthcare_claims.csv')

In [4]:
data.dtypes

Provider_ID                               object
Claim_ID                                  object
Patient_Age                                int64
Patient_Gender                            object
Diagnosis_Code                            object
Procedure_Code                             int64
Claim_Amount                             float64
Approved_Amount                          float64
Insurance_Type                            object
Claim_Submission_Date                     object
Days_Between_Service_and_Claim             int64
Number_of_Claims_Per_Provider_Monthly      int64
Provider_Specialty                        object
Patient_State                             object
Claim_Status                              object
Is_Fraud                                    bool
Length_of_Stay                             int64
Visit_Type                                object
Chronic_Condition_Flag                      bool
Prior_Visits_12m                           int64
dtype: object

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         10000 non-null  object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

Date Type Conversion 

In [6]:
data['Claim_Submission_Date'] = pd.to_datetime(data['Claim_Submission_Date'])

1. NUMERICAL FEATURES

In [12]:
# Ratios: Approval Ratio (How much of the billed claim was approved?)

data['Approval_Ratio'] = np.where(data['Claim_Amount'] > 0, data['Approved_Amount'] / data['Claim_Amount'], 0)

In [ ]:
# B. Differences: Dollar variance between billed and approved amounts

data['Unapproved_Difference'] = data['Claim_Amount'] - data['Approved_Amount']

In [14]:
# C. Aggregations: Provider specialty average & claim-to-average ratio

data['Avg_Claim_By_Specialty'] = data.groupby('Provider_Specialty')['Claim_Amount'].transform('mean')

In [15]:
# Claim-to-Average-Provider Amount (How many times higher than peer average?)

data['Claim_To_Avg_Specialty_Ratio'] = data['Claim_Amount'] / data['Avg_Claim_By_Specialty']

In [16]:
# Historical Provider Statistics (Volume and Historical Fraud Count)

data['Provider_Total_Claim_Volume'] = data.groupby('Provider_ID')['Claim_ID'].transform('count')
data['Provider_Historical_Fraud_Count'] = data.groupby('Provider_ID')['Is_Fraud'].transform('sum')

In [ ]:
# D. Log Transformations: Apply log1p to heavily right-skewed financial columns

data['Log_Claim_Amount'] = np.log1p(data['Claim_Amount'])

2. CATEGORICAL FEATURES

In [18]:
# A. Binning / Buckets: Low / Medium / High Claim Amounts using Quantiles
data['Claim_Amount_Bucket'] = pd.qcut(data['Claim_Amount'], q=3, labels=['Low Claim', 'Medium Claim', 'High Claim'])

In [19]:
# B. Provider Volume Buckets
volume_bins = [-1, 10, 50, np.inf]
volume_labels = ['Low Volume', 'Medium Volume', 'High Volume']
data['Provider_Volume_Category'] = pd.cut(data['Number_of_Claims_Per_Provider_Monthly'], bins=volume_bins, labels=volume_labels)

In [20]:
# C. Claim Delay Buckets
delay_bins = [-1, 7, 30, np.inf]
delay_labels = ['Same Week', 'Standard Delay (8-30 Days)', 'Severe Delay (>30 Days)']
data['Claim_Delay_Category'] = pd.cut(data['Days_Between_Service_and_Claim'], bins=delay_bins, labels=delay_labels)

In [21]:
# D. Handling Rare Categories in Patient_State (Combining rare states into 'Other')
state_counts = data['Patient_State'].value_counts()
rare_states = state_counts[state_counts < 100].index
data['Patient_State_Grouped'] = data['Patient_State'].replace(rare_states, 'Other State')

In [ ]:
# Converting Chronic_Condition_Flag and Is_Fraud to integer binary (0/1)
data['Chronic_Condition_Enc'] = data['Chronic_Condition_Flag'].astype(int)
data['Is_Fraud_Enc'] = data['Is_Fraud'].astype(int)

3. DATE FEATURES

In [23]:
data['Submission_Year'] = data['Claim_Submission_Date'].dt.year
data['Submission_Month'] = data['Claim_Submission_Date'].dt.month
data['Submission_Day'] = data['Claim_Submission_Date'].dt.day
data['Submission_DayOfWeek'] = data['Claim_Submission_Date'].dt.day_name()
data['Is_Weekend_Submission'] = data['Claim_Submission_Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Time Between Events (Service to Submission Gap is already present, but extracting quarter-end gaps)
data['Is_Month_End_Submission'] = data['Claim_Submission_Date'].dt.is_month_end.astype(int)

In [25]:
data.to_csv('fully_engineered_healthcare_data.csv', index=False)
print("Feature Engineering complete! Output saved as 'fully_engineered_healthcare_data.csv'.")

Feature Engineering complete! Output saved as 'fully_engineered_healthcare_data.csv'.


In [26]:
data.head(4)

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,...,Claim_Delay_Category,Patient_State_Grouped,Chronic_Condition_Enc,Is_Fraud_Enc,Submission_Year,Submission_Month,Submission_Day,Submission_DayOfWeek,Is_Weekend_Submission,Is_Month_End_Submission
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,...,Standard Delay (8-30 Days),NY,1,0,2024,9,1,Sunday,1,0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,...,Same Week,IL,1,0,2022,9,5,Monday,0,0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,...,Standard Delay (8-30 Days),IL,1,0,2022,4,11,Monday,0,0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,...,Standard Delay (8-30 Days),TX,0,0,2023,10,11,Wednesday,0,0
